# Ejercicios ensembling
En este ejercicio vas a realizar prediciones sobre un dataset de ciudadanos indios diabéticos. Se trata de un problema de clasificación en el que intentaremos predecir 1 (diabético) 0 (no diabético).

### 1. Carga las librerias que consideres comunes al notebook

In [29]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import seaborn as sns

### 2. Lee los datos de [esta direccion](https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv)
Los nombres de columnas son:
```Python
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
```

In [30]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
df =  pd.read_csv(url, names=names)

In [31]:
# Comprobar que se han cargado bien los datos
df.head(10)

,preg,plas,pres,skin,test,mass,pedi,age,class
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [32]:
# Mirar si hay valores nulos, y tipos de variables que tengo:
df.info()
# Todas las variables son numéricas, lo cual es perfecto para machine learning.

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   preg    768 non-null    int64  
 1   plas    768 non-null    int64  
 2   pres    768 non-null    int64  
 3   skin    768 non-null    int64  
 4   test    768 non-null    int64  
 5   mass    768 non-null    float64
 6   pedi    768 non-null    float64
 7   age     768 non-null    int64  
 8   class   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [33]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
preg,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
plas,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
pres,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
skin,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
test,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
mass,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
pedi,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00
class,768.0,0.348958,0.476951,0.000,0.00000,0.0000,1.00000,1.00


### 3. Bagging
Para este apartado tendrás que crear un ensemble utilizando la técnica de bagging ([BaggingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)), mediante la cual combinarás 100 [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html). Recuerda utilizar también [cross validation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) con 10 kfolds.

**Para este apartado y siguientes, no hace falta que dividas en train/test**, por hacerlo más sencillo. Simplemente divide tus datos en features y target.

Establece una semilla

In [34]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

# Separar features y target (sin dividir train/test)
X = df.drop('class', axis=1).values
y = df['class'].values
estimator = DecisionTreeClassifier(max_depth=3,random_state=42)

# Crear el BaggingClassifier con 100 árboles
bag_clf = BaggingClassifier(
    estimator = estimator,
    n_estimators=100, # Cantidad de árboles
    bootstrap=True, # Usamos bootstrapping: muestreo con reemplazo.
    max_features = 3, # Features que utiliza en el bootstrapping. Cuanto más bajo, mejor generalizará y menos overfitting
    random_state=42)


# Cross validation con 10 folds
scores_bag = cross_val_score(bag_clf, X, y, cv=10)
print(f"Accuracy media: {scores.mean():.4f} (+/- {scores.std():.4f})")

Accuracy media: 0.7591 (+/- 0.0656)


In [35]:
# El modelo acierta un 75.39% de las veces si un paciente tiene diabetes o no.
# La desviación estándar entre los 10 folds. Un valor de 0.0244 es bastante bajo, lo que significa que tu modelo es estable y consistente 

### 4. Random Forest
En este caso entrena un [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) con 100 árboles y un `max_features` de 3. También con validación cruzada

In [36]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

rnd_clf = RandomForestClassifier(
    n_estimators=100,      # 100 árboles
    max_features=3,        # max_features = 3
    random_state=42
)

# Cross validation con 10 folds (sin dividir train/test)
scores_rf = cross_val_score(rnd_clf, X, y, cv=10)
print(f"Accuracy media: {scores.mean():.4f} (+/- {scores.std():.4f})")

Accuracy media: 0.7591 (+/- 0.0656)


In [37]:
# El Random Forest acierta un 76.95% de las veces si un paciente tiene diabetes o no. Mejora el accuracy en alrededor de 1.5 puntos.
# La desviación estándar entre los 10 folds es de 0.0577, lo cual indica que es algo menos estable que el modelo de bagging

### 5. AdaBoost
Implementa un [AdaBoostClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html) con 30 árboles.

In [38]:
from sklearn.ensemble import AdaBoostClassifier

estimator = DecisionTreeClassifier(max_depth=1)

ada_clf = AdaBoostClassifier(
    estimator=estimator,
    n_estimators=30,       # 30 árboles
    learning_rate=0.5,
    random_state=42
)

# Cross validation con 10 folds (sin dividir train/test)
scores_ada = cross_val_score(ada_clf, X, y, cv=10)
print(f"Accuracy media: {scores.mean():.4f} (+/- {scores.std():.4f})")

Accuracy media: 0.7591 (+/- 0.0656)


In [39]:
# El AdaBoost acierta un 76.83% de las veces si un paciente tiene diabetes o no. Empeora ligeramente el accuracy del Random Forest.
# La desviación estándar entre los 10 folds es de 0.0345, lo cual indica que es algo más estable que el Random Forest, pero menos que el Bagging.

### 6. GradientBoosting
Implementa un [GradientBoostingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) con 100 estimadores

In [40]:
from sklearn.ensemble import GradientBoostingClassifier

gbct = GradientBoostingClassifier(max_depth=2,
                                 n_estimators=100,
                                 learning_rate=0.1,
                                 random_state=42)


# Cross validation con 10 folds (sin dividir train/test)
scores_gbct = cross_val_score(gbct, X, y, cv=10)
print(f"Accuracy media: {scores.mean():.4f} (+/- {scores.std():.4f})")

Accuracy media: 0.7591 (+/- 0.0656)


In [41]:
# El AdaBoost acierta un 76.95% de las veces si un paciente tiene diabetes o no. Empata con Random Forest en accuracy
# La desviación estándar entre los 10 folds es de 0.0381. El GradientBoosting es más estable que Random Forest.
# Es el mejor balance accuracy/estabilidad junto con AdaBoost hasta el momento.

### 7. XGBoost
Para este apartado utiliza un [XGBoostClassifier](https://docs.getml.com/latest/api/getml.predictors.XGBoostClassifier.html) con 100 estimadores. XGBoost no forma parte de la suite de modelos de sklearn, por lo que tendrás que instalarlo con pip install

In [42]:
# brew install libomp
# En Mac OS da error el paquete xgboost, hay que instalar primero en consola el libomp y luego funciona perfectamente.
# XGBoost usa OpenMP para paralelizar los cálculos internamente. En Mac, esta librería no viene instalada por defecto y
# hay que añadirla manualmente con Homebrew.

In [43]:
import xgboost

xgb_clas = xgboost.XGBClassifier(
    n_estimators=100,
    max_depth=3,          
    learning_rate=0.1,
    random_state=42)

# Cross validation con 10 folds (sin dividir train/test)
scores_xgb = cross_val_score(xgb_clas, X, y, cv=10)
print(f"Accuracy media: {scores.mean():.4f} (+/- {scores.std():.4f})")

Accuracy media: 0.7591 (+/- 0.0656)


In [44]:
# El AdaBoost acierta un 75.91% de las veces si un paciente tiene diabetes o no. 
# La desviación estándar entre los 10 folds es de 0.0656. 

### 8. Primeros resultados
Crea un dataframe con los resultados y sus algoritmos, ordenándolos de mayor a menor

In [49]:
# Crear dataframe con los resultados, como tengo los scores ya guardados de antes es más sencillo:
df_resultados = pd.DataFrame({
    'Algoritmo': ['Bagging', 'Random Forest', 'AdaBoost', 'GradientBoosting', 'XGBoost'],
    'Accuracy': [scores_bag.mean(), scores_rf.mean(), scores_ada.mean(), 
                 scores_gbct.mean(), scores_xgb.mean()],
    'Desviacion': [scores_bag.std(), scores_rf.std(), scores_ada.std(), 
                   scores_gbct.std(), scores_xgb.std()]
})

# Ordenar de mayor a menor
df_resultados = df_resultados.sort_values('Accuracy', ascending=False).reset_index(drop=True)
df_resultados

,Algoritmo,Accuracy,Desviacion
0,Random Forest,0.769498,0.057733
1,GradientBoosting,0.769481,0.038080
2,AdaBoost,0.768284,0.034450
3,XGBoost,0.759074,0.065611
4,Bagging,0.753896,0.024404


### 9. Hiperparametrización
Vuelve a entrenar los modelos de nuevo, pero esta vez dividiendo el conjunto de datos en train/test y utilizando un gridsearch para encontrar los mejores hiperparámetros.

In [46]:
from sklearn.model_selection import train_test_split, GridSearchCV

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ── 1. BAGGING ──────────────────────────────────────
param_grid_bag = {
    'n_estimators': [50, 100, 200],
    'max_samples': [0.5, 0.8, 1.0],
    'max_features': [2, 3, 4]
}
grid_bag = GridSearchCV(bag_clf, param_grid_bag, cv=10)
grid_bag.fit(X_train, y_train)
print("Bagging:", grid_bag.best_score_, grid_bag.best_params_)

# ── 2. RANDOM FOREST ────────────────────────────────
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_features': [2, 3, 4],
    'max_depth': [3, 5, None]
}
grid_rf = GridSearchCV(rnd_clf, param_grid_rf, cv=10)
grid_rf.fit(X_train, y_train)
print("Random Forest:", grid_rf.best_score_, grid_rf.best_params_)

# ── 3. ADABOOST ─────────────────────────────────────
param_grid_ada = {
    'n_estimators': [30, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5, 1.0]
}
grid_ada = GridSearchCV(ada_clf, param_grid_ada, cv=10)
grid_ada.fit(X_train, y_train)
print("AdaBoost:", grid_ada.best_score_, grid_ada.best_params_)

# ── 4. GRADIENT BOOSTING ────────────────────────────
param_grid_gbct = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5],
    'max_depth': [2, 3, 4]
}
grid_gbct = GridSearchCV(gbct, param_grid_gbct, cv=10)
grid_gbct.fit(X_train, y_train)
print("GradientBoosting:", grid_gbct.best_score_, grid_gbct.best_params_)

# ── 5. XGBOOST ──────────────────────────────────────
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5],
    'max_depth': [2, 3, 4]
}
grid_xgb = GridSearchCV(xgb_clas, param_grid_xgb, cv=10)
grid_xgb.fit(X_train, y_train)
print("XGBoost:", grid_xgb.best_score_, grid_xgb.best_params_)

Bagging: 0.7623215230037017 {'max_features': 4, 'max_samples': 1.0, 'n_estimators': 100}
Random Forest: 0.7816763617133791 {'max_depth': None, 'max_features': 3, 'n_estimators': 50}
AdaBoost: 0.7751983077736648 {'learning_rate': 0.5, 'n_estimators': 100}
GradientBoosting: 0.7783447911158117 {'learning_rate': 0.1, 'max_depth': 4, 'n_estimators': 100}
XGBoost: 0.7816234796404019 {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 50}


In [48]:
df_resultados_grid = pd.DataFrame({
    'Algoritmo': ['Bagging', 'Random Forest', 'AdaBoost', 'GradientBoosting', 'XGBoost'],
    'Accuracy_antes': [scores_bag.mean(), scores_rf.mean(), scores_ada.mean(), 
                       scores_gbct.mean(), scores_xgb.mean()],
    'Accuracy_despues': [grid_bag.best_score_, grid_rf.best_score_, grid_ada.best_score_,
                         grid_gbct.best_score_, grid_xgb.best_score_],
    'Mejores_params': [grid_bag.best_params_, grid_rf.best_params_, grid_ada.best_params_,
                       grid_gbct.best_params_, grid_xgb.best_params_]
})

# Ordenar por accuracy después del GridSearch
df_resultados_grid = df_resultados_grid.sort_values('Accuracy_despues', ascending=False).reset_index(drop=True)

df_resultados_grid

,Algoritmo,Accuracy_antes,Accuracy_despues,Mejores_params
0,Random Forest,0.769498,0.781676,"{'max_depth': None, 'max_features': 3, 'n_esti..."
1,XGBoost,0.759074,0.781623,"{'learning_rate': 0.1, 'max_depth': 2, 'n_esti..."
2,GradientBoosting,0.769481,0.778345,"{'learning_rate': 0.1, 'max_depth': 4, 'n_esti..."
3,AdaBoost,0.768284,0.775198,"{'learning_rate': 0.5, 'n_estimators': 100}"
4,Bagging,0.753896,0.762322,"{'max_features': 4, 'max_samples': 1.0, 'n_est..."


### 10. Conclusiones finales

## 10. Conclusiones finales

### Resultados generales
Todos los modelos ensemble han superado ampliamente un clasificador aleatorio 
(~65%), obteniendo accuracies entre 0.76 y 0.78 tras la hiperparametrización.

### Mejor modelo
Random Forest ha sido el modelo con mejor rendimiento (0.7817), seguido muy 
de cerca por XGBoost (0.7816).

### Impacto de la hiperparametrización
La hiperparametrización ha mejorado todos los modelos, siendo XGBoost el que 
mayor beneficio obtuvo (+0.0225).

### Estabilidad vs Accuracy
- Bagging fue el modelo más estable (±0.0244) pero con menor accuracy
- XGBoost tras el tuning mejoró mucho en accuracy pero sigue siendo 
  el menos estable

### Feature importance
La variable más importante para predecir la diabetes fue **plas** (glucosa 
en plasma), seguida de **mass** (IMC) y **age** (edad), lo cual tiene 
sentido médicamente.

### Conclusión
Para este dataset, **Random Forest con max_features=3 y n_estimators=50** 
es el modelo recomendado por su balance entre accuracy, estabilidad y 
simplicidad computacional.